In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np
DATA_DIR = '/content/drive/MyDrive/nba-xgboost/data'

df = pd.read_csv(f"{DATA_DIR}/games_clean.csv", parse_dates=['GAME_DATE'])
df = df.sort_values(['TEAM_ID','GAME_DATE','GAME_ID']).reset_index(drop=True)

FEATURES = ['PTS','FG_PCT','FG3_PCT','FT_PCT','FGM','FGA','FG3M','FG3A',
            'FTM','FTA','AST','TOV','OREB','DREB','REB','STL','BLK','PF','PLUS_MINUS']
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(23936, 28)


In [2]:
ROLL = [f"R10_{f}" for f in FEATURES]

rolled = df.groupby('TEAM_ID')[FEATURES].transform(
    lambda s: s.shift(1).rolling(window=10).mean()
)
df[ROLL] = rolled.values
print(df[ROLL].isnull().sum().iloc[0], "rows without a full 10-game history")

300 rows without a full 10-game history


In [3]:
before = len(df)
df = df.dropna(subset=ROLL)
print("Dropped", before - len(df), "rows | remaining", df.shape)

Dropped 300 rows | remaining (23636, 47)


In [4]:
row = df.sample(1, random_state=7).iloc[0]
past = df[(df.TEAM_ID == row.TEAM_ID) & (df.GAME_DATE < row.GAME_DATE)].tail(10)

print(row.TEAM_ABBREVIATION, row.GAME_DATE.date())
print("Hand-computed PLUS_MINUS :", round(past['PLUS_MINUS'].mean(), 4))
print("Stored R10_PLUS_MINUS    :", round(row['R10_PLUS_MINUS'], 4))
print("Includes current game?   :", not np.isclose(past['PLUS_MINUS'].mean(), row['R10_PLUS_MINUS']))

DET 2026-03-03
Hand-computed PLUS_MINUS : 9.8
Stored R10_PLUS_MINUS    : 9.8
Includes current game?   : False


In [6]:
keep_home = ['GAME_ID','GAME_DATE','SEASON_ID','TEAM_ID','TEAM_ABBREVIATION','WIN'] + ROLL
keep_away = ['GAME_ID','TEAM_ID','TEAM_ABBREVIATION'] + ROLL

home = df[df.IS_HOME == 1][keep_home].rename(
    columns={'TEAM_ID':'HOME_TEAM_ID','TEAM_ABBREVIATION':'HOME_TEAM','WIN':'HOME_WIN',
             **{c: f"HOME_{c}" for c in ROLL}})
away = df[df.IS_HOME == 0][keep_away].rename(
    columns={'TEAM_ID':'AWAY_TEAM_ID','TEAM_ABBREVIATION':'AWAY_TEAM',
             **{c: f"AWAY_{c}" for c in ROLL}})

m = home.merge(away, on='GAME_ID', how='inner')
print(m.shape)

(11812, 46)


In [7]:
for f in FEATURES:
    m[f"{f}_DIFF"] = m[f"HOME_R10_{f}"] - m[f"AWAY_R10_{f}"]

DIFF = [f"{f}_DIFF" for f in FEATURES]
print(len(DIFF), "differential features")

19 differential features


In [8]:
m = m[m['GAME_DATE'] >= '2017-10-01']
m = m.sort_values(['GAME_DATE','GAME_ID']).reset_index(drop=True)
print(m.shape, m['GAME_DATE'].min().date(), "->", m['GAME_DATE'].max().date())

(10738, 65) 2017-10-17 -> 2026-04-12


In [9]:
print("Games:", len(m), "(expect about 10,750)")
print("Nulls in features:", m[DIFF].isnull().sum().sum(), "(expect 0)")
print("Home win rate:", round(m['HOME_WIN'].mean(), 4))
print(m.groupby('SEASON_ID').size())
print(m[DIFF].describe().T[['mean','std']].round(3))

Games: 10738 (expect about 10,750)
Nulls in features: 0 (expect 0)
Home win rate: 0.5595
SEASON_ID
22017    1230
22018    1230
22019    1059
22020    1080
22021    1230
22022    1230
22023    1229
22024    1225
22025    1225
dtype: int64
                  mean    std
PTS_DIFF        -0.176  7.668
FG_PCT_DIFF     -0.000  0.031
FG3_PCT_DIFF    -0.001  0.041
FT_PCT_DIFF     -0.000  0.057
FGM_DIFF        -0.062  3.164
FGA_DIFF        -0.034  4.306
FG3M_DIFF       -0.040  2.703
FG3A_DIFF       -0.031  6.122
FTM_DIFF        -0.013  3.307
FTA_DIFF        -0.020  4.016
AST_DIFF        -0.058  3.448
TOV_DIFF         0.013  2.201
OREB_DIFF        0.006  2.363
DREB_DIFF       -0.014  3.217
REB_DIFF        -0.008  3.963
STL_DIFF         0.002  1.719
BLK_DIFF        -0.007  1.468
PF_DIFF          0.003  2.458
PLUS_MINUS_DIFF -0.111  9.618


In [10]:
# Full table (keeps IDs, dates, team names and rolling stats)
m.to_csv(f"{DATA_DIR}/matchups_full.csv", index=False)

# Model-ready table: only what XGBoost will use
model_df = m[['GAME_ID','GAME_DATE','HOME_TEAM','AWAY_TEAM'] + DIFF + ['HOME_WIN']]
model_df.to_csv(f"{DATA_DIR}/model_data.csv", index=False)
print(model_df.shape)

(10738, 24)
